In [0]:
import pandas as pd

In [0]:
token = "sp=r&st=2026-03-29T11:36:29Z&se=2026-03-31T19:51:29Z&spr=https&sv=2024-11-04&sr=c&sig=%2F4slZE209JBvkH4ClxTuvwbFid9jqBbTTfzINyroc8U%3D"

file_names =[
{"file":"bulk_rides"},
{"file":"map_cancellation_reasons"},
{"file":"map_cities"},
{"file":"map_payment_methods"},
{"file":"map_ride_statuses"},
{"file":"map_vehicle_makes"},
{"file":"map_vehicle_types"}
]

for file in file_names:
    url = f"https://uberstoragekd.blob.core.windows.net/raw/ingestion/{file["file"]}.json?{token}"

    df = pd.read_json(url)
    df_spark = spark.createDataFrame(df)

    # write data to bronze layer
    df_spark.write.format("delta")\
            .mode("overwrite")\
            .option("overwriteSchema", "true")\
            .saveAsTable(f"ubercat.bronze.{file["file"]}")


In [0]:
%sql
select * from ubercat.bronze.map_cities

city_id,city,state,region,updated_at
1,New York,NY,Northeast,2026-03-30T00:24:07.377Z
2,Los Angelas,CA,West,2026-03-30T00:24:07.377Z
3,Chicago,IL,Midwest,2026-03-30T00:24:07.377Z
4,Houston,TX,South,2026-03-30T00:24:07.377Z
5,Phoenix,AZ,Southwest,2026-03-30T00:24:07.377Z
6,Philadelphia,PA,Northeast,2026-03-30T00:24:07.377Z
7,San Antonio,TX,South,2026-03-30T00:24:07.377Z
8,San Diego,CA,West,2026-03-30T00:24:07.377Z
9,Dallas,TX,South,2026-03-30T00:24:07.377Z
10,San Jose,CA,West,2026-03-30T00:24:07.377Z


In [0]:
%sql
select * from ubercat.bronze.rides_raw

key,value,topic,partition,offset,timestamp,timestampType,rides
null,eyJyaWRlX2lkIjogImNjMDVmMmQ3LTcxY2QtNDRkMS05MjM5LWU0OGEwOGE1ZWYzNCIsICJjb25maXJtYXRpb25fbnVtYmVyIjogImNUMC00NjM2LVN6NDIiLCAicGFzc2VuZ2VyX2lkIjogImRmMzVkZWU1LTE4YzAtNGE4Yi1hYzViLWY5MmQ5NzI0ZDdlMCIsICJkcml2ZXJfaWQiOiAiNmNmMjA5MzEtZmRlNy00MTM1LWIzNmQtZmY1N2Y2OWI4YTdhIiwgInZlaGljbGVfaWQiOiAiOTNkODVjMjAtMGU1OS00NzgxLWIxNWEtY2ExNjJiNjVkN2ZlIiwgInBpY2t1cF9sb2NhdGlvbl9pZCI6ICJlMzc0ZTNhMy1lMWI3LTRjNTktYmRhNC0yNDUwMWJjZTBhOTEiLCAiZHJvcG9mZl9sb2NhdGlvbl9pZCI6ICI4Y2U5MTZkOC02ZGUzLTRmNTQtOTRkOC1mMDQ0MzhkODJhZmQiLCAidmVoaWNsZV90eXBlX2lkIjogMSwgInZlaGljbGVfbWFrZV9pZCI6IDYsICJwYXltZW50X21ldGhvZF9pZCI6IDIsICJyaWRlX3N0YXR1c19pZCI6IDEsICJwaWNrdXBfY2l0eV9pZCI6IDcsICJkcm9wb2ZmX2NpdHlfaWQiOiA5LCAiY2FuY2VsbGF0aW9uX3JlYXNvbl9pZCI6IDQsICJwYXNzZW5nZXJfbmFtZSI6ICJKdWxpYSBTdGV2ZW5zb24iLCAicGFzc2VuZ2VyX2VtYWlsIjogInRoZW5zb25AZXhhbXBsZS5jb20iLCAicGFzc2VuZ2VyX3Bob25lIjogIjg4NC00MDEtMjEwMyIsICJkcml2ZXJfbmFtZSI6ICJNci4gSm9obiBKb2huc29uIERWTSIsICJkcml2ZXJfcmF0aW5nIjogNC41MywgImRyaXZlcl9waG9uZSI6ICIrMS05NzMtNzgzLTA1MTR4NzA2OTUiLCAiZHJpdmVyX2xpY2Vuc2UiOiAiY1MtYXJrLTU0NDY0MzYiLCAidmVoaWNsZV9tb2RlbCI6ICJDb21tdW5pdHkiLCAidmVoaWNsZV9jb2xvciI6ICJXaGl0ZSIsICJsaWNlbnNlX3BsYXRlIjogIklCUi02ODc5IiwgInBpY2t1cF9hZGRyZXNzIjogIjkwNjIgTm9ydG9uIExha2UgQXB0LiA2OTQsIERvdWdsYXNtb3V0aCwgTVAgOTI1MzUiLCAicGlja3VwX2xhdGl0dWRlIjogODAuNzk1MzM0LCAicGlja3VwX2xvbmdpdHVkZSI6IDkyLjg2NzU1MywgImRyb3BvZmZfYWRkcmVzcyI6ICIzNDMwIEFuZHJldyBMb2NrcywgUGF0cmljaWFoYXZlbiwgV1kgNDA3MDQiLCAiZHJvcG9mZl9sYXRpdHVkZSI6IC04MS43MTU0MDEsICJkcm9wb2ZmX2xvbmdpdHVkZSI6IDY5LjA3Mjg3MiwgImRpc3RhbmNlX21pbGVzIjogMi4yNywgImR1cmF0aW9uX21pbnV0ZXMiOiAxMDQsICJib29raW5nX3RpbWVzdGFtcCI6ICIyMDI2LTAzLTA3VDAwOjQ5OjE0LjIwMjUyNiIsICJwaWNrdXBfdGltZXN0YW1wIjogIjIwMjYtMDMtMDdUMDA6NTE6MTQuMjAyNTI2IiwgImRyb3BvZmZfdGltZXN0YW1wIjogIjIwMjYtMDMtMDdUMDI6MzU6MTQuMjAyNTI2IiwgImJhc2VfZmFyZSI6IDIuNSwgImRpc3RhbmNlX2ZhcmUiOiAzLjk3LCAidGltZV9mYXJlIjogMzYuNCwgInN1cmdlX211bHRpcGxpZXIiOiAyLjAzLCAic3VidG90YWwiOiA4Ny4wMywgInRpcF9hbW91bnQiOiAzLCAidG90YWxfZmFyZSI6IDkwLjAzLCAicmF0aW5nIjogbnVsbH0=,engevent,0,0,2026-03-29T00:51:15.442Z,0,"{""ride_id"": ""cc05f2d7-71cd-44d1-9239-e48a08a5ef34"", ""confirmation_number"": ""cT0-4636-Sz42"", ""passenger_id"": ""df35dee5-18c0-4a8b-ac5b-f92d9724d7e0"", ""driver_id"": ""6cf20931-fde7-4135-b36d-ff57f69b8a7a"", ""vehicle_id"": ""93d85c20-0e59-4781-b15a-ca162b65d7fe"", ""pickup_location_id"": ""e374e3a3-e1b7-4c59-bda4-24501bce0a91"", ""dropoff_location_id"": ""8ce916d8-6de3-4f54-94d8-f04438d82afd"", ""vehicle_type_id"": 1, ""vehicle_make_id"": 6, ""payment_method_id"": 2, ""ride_status_id"": 1, ""pickup_city_id"": 7, ""dropoff_city_id"": 9, ""cancellation_reason_id"": 4, ""passenger_name"": ""Julia Stevenson"", ""passenger_email"": ""thenson@example.com"", ""passenger_phone"": ""884-401-2103"", ""driver_name"": ""Mr. John Johnson DVM"", ""driver_rating"": 4.53, ""driver_phone"": ""+1-973-783-0514x70695"", ""driver_license"": ""cS-ark-5446436"", ""vehicle_model"": ""Community"", ""vehicle_color"": ""White"", ""license_plate"": ""IBR-6879"", ""pickup_address"": ""9062 Norton Lake Apt. 694, Douglasmouth, MP 92535"", ""pickup_latitude"": 80.795334, ""pickup_longitude"": 92.867553, ""dropoff_address"": ""3430 Andrew Locks, Patriciahaven, WY 40704"", ""dropoff_latitude"": -81.715401, ""dropoff_longitude"": 69.072872, ""distance_miles"": 2.27, ""duration_minutes"": 104, ""booking_timestamp"": ""2026-03-07T00:49:14.202526"", ""pickup_timestamp"": ""2026-03-07T00:51:14.202526"", ""dropoff_timestamp"": ""2026-03-07T02:35:14.202526"", ""base_fare"": 2.5, ""distance_fare"": 3.97, ""time_fare"": 36.4, ""surge_multiplier"": 2.03, ""subtotal"": 87.03, ""tip_amount"": 3, ""total_fare"": 90.03, ""rating"": null}"
null,eyJyaWRlX2lkIjogIjRhNGRhNThlLTBjYmItNDliNi1iMDUxLTg0YmQ5MzMwNTMzYiIsICJjb25maXJtYXRpb25fbnVtYmVyIjogIlBDOS02NzM5LVRMNzQiLCAicGFzc2VuZ2VyX2lkIjogIjM2MTRjOTZlLTFiZjMtNGVjYy1hNThjLTEzN2NlNzJhNDUwOCIsICJkcml2ZXJfaWQiOiAiY2M2OGNkZmUtM2RkMy00YjFiLWFjNWEtNWM1NGU2